In [ ]:
rdm_url = 'http://localhost:5000/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-Migration-S3'
target_storage_name = 'Amazon S3'
target_file_name = '移行前 + データ.txt'
target_file_content = 'S3 storage migration fixture'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
new_file_name = None
new_file_content = 'Created after S3 storage migration'

In [ ]:
from getpass import getpass
from pathlib import Path
from tempfile import mkdtemp
import sys

if idp_username_1 is None:
    idp_username_1 = input('Username: ')
if idp_password_1 is None:
    idp_password_1 = getpass(f'Password for {idp_username_1}: ')

sys.path.insert(0, str(Path.cwd().parent))
from playwright.async_api import expect
from scripts import grdm
from scripts.playwright import init_pw_context, run_pw, finish_pw_context

work_dir = Path(mkdtemp())
await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path or str(work_dir))
work_dir

# 版の切替後に既存のS3接続と保存データを使う

- サブシステム名: ストレージ
- ページ/アドオン: Amazon S3
- 機能分類: 移行互換性
- 用意するテストデータ: 移行試験用GRDM、ユーザー、S3バケット
- シナリオ名: 版の切替後のS3接続とファイルの利用

## GRDMトップページを開く

ログイン画面へ移動できること。

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent = page.get_by_role('button', name='同意する', exact=True)
    sign_in = page.locator('[data-test-sign-in-button]')
    await expect(page.locator('button:visible').and_(consent).or_(sign_in).first).to_be_visible(
        timeout=transition_timeout)
    if await consent.is_visible():
        await consent.click()
    await expect(page.locator('[data-test-sign-in-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## GRDMにログインする

マイプロジェクトが表示されること。

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('[data-test-create-project-modal-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## マイプロジェクトから対象のプロジェクトを開く

プロジェクトのファイルページを開けること。

In [ ]:
async def _step(page):
    await page.locator('[data-test-dashboard-item-title]').filter(has_text=rdm_project_name).click()
    await expect(page.locator('#projectNavFiles a')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 上部メニューから「ファイル」をクリックする

既存のAmazon S3接続先が表示されること。

In [ ]:
async def _step(page):
    await page.locator('#projectNavFiles a').click()
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(
        timeout=transition_timeout)

await run_pw(_step)

## 既存ファイルを選び、ツールバーの「ダウンロード」をクリックする

保存済みファイルをダウンロードでき、内容が一致すること。

In [ ]:
async def _step(page):
    await grdm.get_select_file_extension_locator(page, target_file_name).click()
    button = page.locator('.fangorn-toolbar-icon').get_by_text('ダウンロード', exact=True)
    async with page.expect_download(timeout=transition_timeout) as pending_download:
        await button.click()
    path = await (await pending_download.value).path()
    assert Path(path).read_text(encoding='utf-8') == target_file_content, '保存済みファイルの内容が一致しません'

await run_pw(_step)

## (指定された場合) 既存のAmazon S3接続に新しいファイルをアップロードする

ファイルがアップロードされ、一覧に表示されること。

In [ ]:
async def _step(page):
    if new_file_name is None:
        return

    path = work_dir / new_file_name
    path.write_text(new_file_content, encoding='utf-8')
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(path))
    await grdm.wait_for_uploaded(page, new_file_name, timeout=transition_timeout)

await run_pw(_step)

In [ ]:
await finish_pw_context()